# pruning-resnet18 — Training trên Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NhatMinh1503/pruning-resnet18/blob/main/training/colab_training.ipynb)

Notebook dùng chung của nhóm để chạy `training/train.py` (baseline) và `training/run_pruning.py` (cắt + fine-tune).

**Chuẩn bị 1 lần:**
1. Google Drive → *Được chia sẻ với tôi* → chuột phải `pruning-project` → **Sắp xếp → Thêm lối tắt → Drive của tôi**.
2. Colab: **Runtime → Change runtime type → T4 GPU** (notebook này đã đặt sẵn GPU).

| Bước | Thời gian (T4, đo ngày 12/10) |
|---|---|
| Baseline 100 epoch | ~47 phút (~28 giây/epoch) → acc **95.10%** |
| Pruning 2 method × 9 mức × 10 epoch | ~1.5 giờ (ước tính) |

Bị ngắt kết nối thì chạy lại từ ô 1: các script tự học tiếp / bỏ qua phần đã xong.

## 1. Kết nối Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CKPT = '/content/drive/MyDrive/pruning-project/checkpoints'
RESULTS = '/content/drive/MyDrive/pruning-project/results'
!ls /content/drive/MyDrive/pruning-project

## 2. Lấy code mới nhất + kiểm tra GPU

In [ ]:
import os
if os.path.isdir('/content/pruning-resnet18'):
    %cd /content/pruning-resnet18
    !git pull
else:
    !git clone https://github.com/NhatMinh1503/pruning-resnet18.git /content/pruning-resnet18
    %cd /content/pruning-resnet18
!pip install -q thop
!nvidia-smi --query-gpu=name --format=csv
# Lấy accuracy.csv chung từ Drive (nếu có) để không ghi đè kết quả cũ
!test -f {RESULTS}/accuracy.csv && cp {RESULTS}/accuracy.csv results/; cat results/accuracy.csv

## 3A. Train baseline (chỉ chạy nếu chưa có `resnet18_baseline.pt`)

Đã có baseline từ 12/10 (95.10%) → **bỏ qua ô này**.

In [ ]:
if os.path.exists(f'{CKPT}/resnet18_baseline.pt'):
    print('Đã có baseline -> bỏ qua')
else:
    !python training/train.py --data-dir /content/data --ckpt-dir {CKPT}

## 3B. Cắt + fine-tune

Chạy từng method để mỗi lần ~45 phút. Cặp nào đã xong sẽ tự bỏ qua.

In [ ]:
!python training/run_pruning.py --data-dir /content/data --ckpt-dir {CKPT} --methods local

In [ ]:
!python training/run_pruning.py --data-dir /content/data --ckpt-dir {CKPT} --methods global

## 4. Lưu kết quả lên Drive chung

In [ ]:
!cp results/accuracy.csv {RESULTS}/
!cat {RESULTS}/accuracy.csv
!ls -lh {CKPT}